In [1]:
# Cell 1 — Setup
!git clone https://github.com/trungcandygit/Nh-m-10tr_devided_by_5.git /tmp/tnbike
!apt-get install -y unrar -q
!pip install rarfile -q

Cloning into '/tmp/tnbike'...
remote: Enumerating objects: 590, done.
remote: Counting objects: 100% (374/374), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 590 (delta 201), reused 237 (delta 133), pack-reused 216 (from 1)
Receiving objects: 100% (590/590), 26.41 MiB | 22.00 MiB/s, done.
Resolving deltas: 100% (231/231), done.
Reading package lists...
Building dependency tree...
Reading state information...
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [2]:
# Cell 2 — Kiểm tra
import os
for f in os.listdir("/tmp/tnbike/de_thi"):
    print(f)

Tnbike_Database_Schema_Doc.docx
tnbike_pdfs_mar2026.rar
01_create_tables.sql
tnbike_emails_mar2026.rar
02_import_data.sql
Database (4).zip
Emails & Files (3).zip
DataExplorers2026 - Đề thi Vòng 2 (2).pdf


In [3]:
# Cell 3 — Giải nén email
import rarfile, os
rarfile.UNRAR_TOOL = "unrar"
os.makedirs("/tmp/tnbike_emails", exist_ok=True)
with rarfile.RarFile("/tmp/tnbike/de_thi/tnbike_emails_mar2026.rar") as rf:
    rf.extractall("/tmp/tnbike_emails/")
print(f"✅ {len(os.listdir('/tmp/tnbike_emails'))} emails")

✅ 1132 emails


In [4]:
# Cell 4 — Đọc 1 email mẫu xem cấu trúc
import email
from email import policy
import os

sample = sorted(os.listdir("/tmp/tnbike_emails"))[0]
print("File mẫu:", sample)

with open(f"/tmp/tnbike_emails/{sample}", "rb") as f:
    msg = email.message_from_bytes(f.read(), policy=policy.default)

print("Subject:", msg["subject"])
print("From   :", msg["from"])
print("Date   :", msg["date"])
print("\n--- BODY ---")
if msg.is_multipart():
    for part in msg.walk():
        if part.get_content_type() == "text/plain":
            print(part.get_content()[:1000])
            break
else:
    print(msg.get_content()[:1000])

File mẫu: BH26_0935.eml
Subject: [ĐẶT HÀNG] BH26.0935
From   : CÔNG TY TNHH THƯƠNG MẠI LONG PHÚ <dathang@longphu.vn>
Date   : Sun, 01 Mar 2026 14:28:41 +0700

--- BODY ---
Kính gửi anh/chị Phòng Kinh Doanh,

Chúng tôi trân trọng gửi yêu cầu đặt hàng lần này. Đơn hàng BH26.0935 ngày 01/03/2026:

  Đại lý   : CÔNG TY TNHH THƯƠNG MẠI LONG PHÚ
  MST      : 167397253
  Địa chỉ  : phố Tràng Thi, phường Hàng Trống, Quận Hoàn Kiếm, TP Hà Nội
  Liên hệ  : 0875649429

File PDF đính kèm gồm chi tiết 1 sản phẩm,
tổng 1 chiếc, trị giá 1.898.148 đồng.

Mong sớm nhận phản hồi xác nhận đơn hàng.

Kính chào và mong nhận phản hồi sớm,
CÔNG TY TNHH THƯƠNG MẠI LONG PHÚ



In [5]:
# Cell 5 — Parse toàn bộ 1132 emails → DataFrame
import email, re, os
import pandas as pd
from email import policy

email_dir = "/tmp/tnbike_emails"
records = []

for fname in os.listdir(email_dir):
    if not fname.endswith(".eml"):
        continue

    with open(f"{email_dir}/{fname}", "rb") as f:
        msg = email.message_from_bytes(f.read(), policy=policy.default)

    # Lấy body
    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                body = part.get_content()
                break
    else:
        body = msg.get_content()

    # Extract các trường
    ma_don   = re.search(r'Đơn hàng\s+(BH26[\.\d]+)\s+ngày', body)
    ngay     = re.search(r'ngày\s+(\d{2}/\d{2}/\d{4})', body)
    dai_ly   = re.search(r'Đại lý\s*:\s*(.+)', body)
    so_sp    = re.search(r'chi tiết\s+(\d+)\s+sản phẩm', body)
    so_chiec = re.search(r'tổng\s+(\d+)\s+chiếc', body)
    tri_gia  = re.search(r'trị giá\s+([\d\.,]+)\s+đồng', body)

    records.append({
        "file"     : fname,
        "ma_don"   : ma_don.group(1)   if ma_don   else None,
        "ngay"     : ngay.group(1)     if ngay     else None,
        "dai_ly"   : dai_ly.group(1).strip() if dai_ly else None,
        "so_sp"    : int(so_sp.group(1))    if so_sp    else None,
        "so_chiec" : int(so_chiec.group(1)) if so_chiec else None,
        "tri_gia"  : tri_gia.group(1).replace(".", "").replace(",", "") if tri_gia else None,
    })

df = pd.DataFrame(records)
df["tri_gia"] = pd.to_numeric(df["tri_gia"], errors="coerce")
df["ngay"]    = pd.to_datetime(df["ngay"], format="%d/%m/%Y", errors="coerce")

print(df.head(5).to_string())
print(f"\n✅ Tổng: {len(df)} emails")
print(f"❌ Thiếu trị giá: {df['tri_gia'].isna().sum()}")
print(f"❌ Thiếu ngày   : {df['ngay'].isna().sum()}")
print(f"\nTổng doanh thu tháng 3: {df['tri_gia'].sum():,.0f} đồng")

            file     ma_don       ngay                        dai_ly  so_sp  so_chiec      tri_gia
0  BH26_1932.eml       None        NaT                          None    NaN       NaN          NaN
1  BH26_1865.eml       None 2026-03-27                          None    NaN       NaN          NaN
2  BH26_1014.eml       None        NaT                          None    NaN       NaN          NaN
3  BH26_1131.eml  BH26.1131 2026-03-09  DOANH NGHIỆP TW TRƯỜNG THỊNH   24.0      85.0  184852746.0
4  BH26_1835.eml       None        NaT                          None    NaN       NaN          NaN

✅ Tổng: 1132 emails
❌ Thiếu trị giá: 773
❌ Thiếu ngày   : 395

Tổng doanh thu tháng 3: 13,637,341,203 đồng


In [6]:
# Cell 6 — Xem body của email bị miss để fix regex
miss = df[df["tri_gia"].isna()].head(3)

for _, row in miss.iterrows():
    fname = row["file"]
    with open(f"{email_dir}/{fname}", "rb") as f:
        msg = email.message_from_bytes(f.read(), policy=policy.default)

    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                body = part.get_content()
                break
    else:
        body = msg.get_content()

    print(f"\n{'='*60}")
    print(f"FILE: {fname}")
    print('='*60)
    print(body[:800])


FILE: BH26_1932.eml
Kính gửi Phòng Kinh Doanh Công ty Xe đạp Thống Nhất,

Nhân dịp cuối tháng, chúng tôi gửi đơn đặt hàng bổ sung.

Chi tiết đơn hàng (xem file đính kèm để biết đầy đủ):

  Số chứng từ  : BH26.1932
  Ngày đặt     : 29/03/2026
  Khách hàng   : DOANH NGHIỆP TW XUÂN MAI
  MST          : 242402965
  Địa chỉ      : Phường Việt Hòa, TP Hải Phòng

  Tổng sản phẩm : 10 dòng
  Tổng số lượng : 31 chiếc
  Tổng giá trị  : 48.565.357 đồng

Vui lòng xác nhận đơn hàng và thông báo thời gian giao hàng.

Xin chân thành cảm ơn,
DOANH NGHIỆP TW XUÂN MAI
dathang@xuanmai.vn
Tel: 0922237365


FILE: BH26_1865.eml
Dear Sales Team,

Để chuẩn bị cho mùa hè 2026, chúng tôi đặt số lượng như sau.

Đính kèm đơn đặt hàng BH26.1865 ngày 27/03/2026.

Thông tin đại lý:
  Tên     : CỬA HÀNG XE ĐẠP QUANG VINH
  MST     : 232799936
  Địa chỉ : Xã Hợp Lý, Phú Thọ
  Tel     : 0967121502

Gồm 8 mặt hàng, 29 chiếc,
tổng giá trị 45.345.079 VNĐ (chưa VAT).

Trân trọng kính chào,
CỬA HÀNG XE ĐẠP QUANG VINH
datha

In [7]:
# Cell 7 — Fix regex bắt cả 3 format

import email, re, os
import pandas as pd
from email import policy

email_dir = "/tmp/tnbike_emails"
records = []

for fname in os.listdir(email_dir):
    if not fname.endswith(".eml"):
        continue

    with open(f"{email_dir}/{fname}", "rb") as f:
        msg = email.message_from_bytes(f.read(), policy=policy.default)

    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                body = part.get_content()
                break
    else:
        body = msg.get_content()

    # ── Mã đơn hàng ──────────────────────────────────────────
    ma_don = re.search(r'BH26[\.\_](\d+)', body)

    # ── Ngày — nhiều format ───────────────────────────────────
    ngay = (
        re.search(r'ngày\s+(\d{2}/\d{2}/\d{4})', body) or
        re.search(r'Ngày đặt\s*:\s*(\d{2}/\d{2}/\d{4})', body) or
        re.search(r'đặt hàng\s+BH26[\.\_]\d+\s+ngày\s+(\d{2}/\d{2}/\d{4})', body)
    )

    # ── Đại lý — nhiều format ─────────────────────────────────
    dai_ly = (
        re.search(r'Đại lý\s*:\s*(.+)', body) or
        re.search(r'Tên\s*:\s*(.+)', body) or
        re.search(r'Khách hàng\s*:\s*(.+)', body)
    )

    # ── Trị giá — nhiều format ────────────────────────────────
    tri_gia = (
        re.search(r'trị giá\s+([\d\.,]+)\s+đồng', body) or
        re.search(r'tổng giá trị\s+([\d\.,]+)\s+VNĐ', body, re.IGNORECASE) or
        re.search(r'Tổng giá trị\s*:\s*([\d\.,]+)\s+đồng', body) or
        re.search(r'Tổng giá trị\s*:\s*([\d\.,]+)', body)
    )

    # ── Số chiếc ─────────────────────────────────────────────
    so_chiec = (
        re.search(r'tổng\s+(\d+)\s+chiếc', body) or
        re.search(r'(\d+)\s+chiếc', body) or
        re.search(r'Tổng số lượng\s*:\s*(\d+)', body)
    )

    def clean_so(s):
        return s.replace(".", "").replace(",", "").strip() if s else None

    records.append({
        "file"    : fname,
        "ma_don"  : f"BH26.{ma_don.group(1)}" if ma_don else None,
        "ngay"    : ngay.group(1)              if ngay    else None,
        "dai_ly"  : dai_ly.group(1).strip()    if dai_ly  else None,
        "so_chiec": int(so_chiec.group(1))     if so_chiec else None,
        "tri_gia" : clean_so(tri_gia.group(1)) if tri_gia  else None,
    })

df = pd.DataFrame(records)
df["tri_gia"] = pd.to_numeric(df["tri_gia"], errors="coerce")
df["ngay"]    = pd.to_datetime(df["ngay"], format="%d/%m/%Y", errors="coerce")

print(df.head(5).to_string())
print(f"\n✅ Tổng emails      : {len(df)}")
print(f"❌ Thiếu trị giá   : {df['tri_gia'].isna().sum()}")
print(f"❌ Thiếu ngày      : {df['ngay'].isna().sum()}")
print(f"\n📅 Phân bố theo ngày:")
print(df["ngay"].value_counts().sort_index())
print(f"\n💰 Tổng doanh thu tháng 3: {df['tri_gia'].sum():,.0f} đồng")

            file     ma_don       ngay                              dai_ly  so_chiec    tri_gia
0  BH26_1932.eml  BH26.1932 2026-03-29            DOANH NGHIỆP TW XUÂN MAI        31   48565357
1  BH26_1865.eml  BH26.1865 2026-03-27          CỬA HÀNG XE ĐẠP QUANG VINH        29   45345079
2  BH26_1014.eml  BH26.1014 2026-03-03               HỘ KINH DOANH VẠN LỢI        35   62442770
3  BH26_1131.eml  BH26.1131 2026-03-09        DOANH NGHIỆP TW TRƯỜNG THỊNH        85  184852746
4  BH26_1835.eml  BH26.1835 2026-03-27  CÔNG TY CỔ PHẦN THƯƠNG MẠI TRÍ ĐỨC         3    3198331

✅ Tổng emails      : 1132
❌ Thiếu trị giá   : 0
❌ Thiếu ngày      : 0

📅 Phân bố theo ngày:
ngay
2026-03-01     28
2026-03-02     33
2026-03-03     21
2026-03-04     22
2026-03-05     30
2026-03-06     18
2026-03-07     38
2026-03-08      1
2026-03-09     50
2026-03-10     26
2026-03-11     33
2026-03-12     30
2026-03-13     29
2026-03-14     39
2026-03-15      2
2026-03-16     43
2026-03-17     16
2026-03-18     35
20

In [9]:
# Cell 10b — Tìm DB ở mọi nơi
import os, glob

# Tìm toàn bộ file trong repo
print("=== Tất cả file trong /tmp/tnbike ===")
for root, dirs, files in os.walk("/tmp/tnbike"):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f"  {fpath}  ({size:,} bytes)")

=== Tất cả file trong /tmp/tnbike ===
  /tmp/tnbike/tnbike_bao_cao_toan_bo.pdf  (148,740 bytes)
  /tmp/tnbike/README.md  (17,645 bytes)
  /tmp/tnbike/requirements.txt  (450 bytes)
  /tmp/tnbike/.gitignore  (34 bytes)
  /tmp/tnbike/1. Kham Pha Du Lieu Ban Hang TNBike/config.yaml  (490 bytes)
  /tmp/tnbike/1. Kham Pha Du Lieu Ban Hang TNBike/02_lam_sach_du_lieu.ipynb  (4,123 bytes)
  /tmp/tnbike/6. Phan Nhom Khach Hang RFM/03_dashboard.ipynb  (4,452 bytes)
  /tmp/tnbike/6. Phan Nhom Khach Hang RFM/config.yaml  (528 bytes)
  /tmp/tnbike/de_thi/Tnbike_Database_Schema_Doc.docx  (22,741 bytes)
  /tmp/tnbike/de_thi/tnbike_pdfs_mar2026.rar  (3,219,214 bytes)
  /tmp/tnbike/de_thi/01_create_tables.sql  (17,522 bytes)
  /tmp/tnbike/de_thi/tnbike_emails_mar2026.rar  (5,337,970 bytes)
  /tmp/tnbike/de_thi/02_import_data.sql  (2,690,529 bytes)
  /tmp/tnbike/de_thi/Database (4).zip  (203,613 bytes)
  /tmp/tnbike/de_thi/Emails & Files (3).zip  (8,508,645 bytes)
  /tmp/tnbike/de_thi/DataExplorers2026 -

In [10]:
# Cell 11 — Giải nén Database zip + xem schema
import zipfile, sqlite3, glob, os

zip_path = "/tmp/tnbike/de_thi/Database (4).zip"
os.makedirs("/tmp/tnbike_db", exist_ok=True)

with zipfile.ZipFile(zip_path) as zf:
    print("Files trong ZIP:")
    for name in zf.namelist():
        print(f"  - {name}")
    zf.extractall("/tmp/tnbike_db/")

# Tìm DB
db_files = glob.glob("/tmp/tnbike_db/**/*", recursive=True)
db_files = [f for f in db_files if os.path.isfile(f)]
print("\nFiles sau giải nén:", db_files)

Files trong ZIP:
  - 01_create_tables.sql
  - 02_import_data.sql
  - Tnbike_Database_Schema_Doc.docx

Files sau giải nén: ['/tmp/tnbike_db/Tnbike_Database_Schema_Doc.docx', '/tmp/tnbike_db/01_create_tables.sql', '/tmp/tnbike_db/02_import_data.sql']


In [11]:
# Cell 13 — Đọc schema từ SQL để hiểu cấu trúc
with open("/tmp/tnbike_db/01_create_tables.sql", "r", encoding="utf-8") as f:
    sql_create = f.read()

print(sql_create[:3000])

-- ============================================================
-- TNBIKE SALES DATABASE — DDL
-- Schema: tnbike
-- PostgreSQL 14+
-- ============================================================
-- Chạy: psql -U postgres -d tnbike_db -f 01_create_tables.sql
-- ============================================================

-- Tạo database (chạy với user postgres nếu chưa có):
-- CREATE DATABASE tnbike_db ENCODING 'UTF8' LC_COLLATE 'vi_VN.UTF-8';
-- \c tnbike_db

CREATE SCHEMA IF NOT EXISTS tnbike;
SET search_path TO tnbike, public;

-- Extension hỗ trợ uuid (dùng cho audit)
CREATE EXTENSION IF NOT EXISTS "uuid-ossp";


-- ============================================================
-- 1. NHÓM SẢN PHẨM CẤP 1  (product_group)
-- ============================================================
CREATE TABLE product_group (
    group_code      VARCHAR(30)     PRIMARY KEY,
    group_name      VARCHAR(100)    NOT NULL,
    description     TEXT,
    created_at      TIMESTAMPTZ     DEFAULT NOW()
);



In [12]:
# Cell 15 — Đọc toàn bộ schema để hiểu tất cả bảng
with open("/tmp/tnbike_db/01_create_tables.sql", "r", encoding="utf-8") as f:
    sql = f.read()

# In tên các bảng
import re
tables = re.findall(r'CREATE TABLE\s+(\w+)\s*\(', sql)
print("📋 Tất cả bảng trong schema:")
for t in tables:
    print(f"  - {t}")

📋 Tất cả bảng trong schema:
  - product_group
  - product_line
  - product
  - product_price
  - province
  - customer
  - sales_order
  - order_line
  - fact_sales


In [14]:
# Cell 16b — Debug: tìm statement bị lỗi
import sqlite3, re

db_path = "/tmp/tnbike_db/tnbike.db"

with open("/tmp/tnbike_db/01_create_tables.sql", "r", encoding="utf-8") as f:
    sql = f.read()

# Adapt PostgreSQL → SQLite
sql = re.sub(r'CREATE SCHEMA.*?;', '', sql, flags=re.DOTALL)
sql = re.sub(r'SET search_path.*?;', '', sql)
sql = re.sub(r'CREATE EXTENSION.*?;', '', sql)
sql = re.sub(r'COMMENT ON.*?;', '', sql, flags=re.DOTALL)
sql = re.sub(r'CREATE INDEX.*?;', '', sql)
sql = re.sub(r'TIMESTAMPTZ', 'TEXT', sql)
sql = re.sub(r'SERIAL', 'INTEGER', sql)
sql = re.sub(r'BOOLEAN', 'INTEGER', sql)
sql = re.sub(r'DEFAULT NOW\(\)', "DEFAULT (datetime('now'))", sql)
sql = re.sub(r'REFERENCES\s+\w+\s*\([^)]+\)', '', sql)
sql = re.sub(r'UNIQUE\s*\([^)]+\)', '', sql)
sql = re.sub(r'NUMERIC\(\d+,\d+\)', 'REAL', sql)
sql = re.sub(r'VARCHAR\(\d+\)', 'TEXT', sql)

# Chạy từng statement để tìm lỗi
conn = sqlite3.connect(":memory:")
statements = [s.strip() for s in sql.split(';') if s.strip()]
for i, stmt in enumerate(statements):
    try:
        conn.execute(stmt)
    except Exception as e:
        print(f"❌ Statement #{i}: {e}")
        print(f"   SQL: {stmt[:300]}")
        print()
conn.close()
print("✅ Scan xong!")

❌ Statement #2: near ")": syntax error
   SQL: -- ============================================================
-- 2. DÒNG SẢN PHẨM CẤP 3  (product_line)
-- ============================================================
CREATE TABLE product_line (
    line_id         INTEGER          PRIMARY KEY,
    line_name       TEXT    NOT NULL,
    group_code

❌ Statement #5: near "giá": syntax error
   SQL: giá thực tế giao dịch nằm trong order_line.
CREATE TABLE product_price (
    price_id        INTEGER          PRIMARY KEY,
    product_code    TEXT     NOT NULL ,
    unit_price      REAL   NOT NULL CHECK (unit_price > 0),
    effective_from  DATE            NOT NULL,
    effective_to    DATE,      

❌ Statement #8: near "FROM": syntax error
   SQL: -- ============================================================
-- 7. ĐẦU PHIẾU CHỨNG TỪ BÁN HÀNG  (sales_order)
-- ============================================================
CREATE TABLE sales_order (
    order_id            INTEGER          PRIMA

In [15]:
# Cell 16c — Fix toàn bộ và tạo DB

import sqlite3, re

db_path = "/tmp/tnbike_db/tnbike.db"

with open("/tmp/tnbike_db/01_create_tables.sql", "r", encoding="utf-8") as f:
    sql = f.read()

# ── Bỏ hoàn toàn PostgreSQL-only blocks ──────────────────────
sql = re.sub(r'CREATE SCHEMA.*?;', '', sql, flags=re.DOTALL)
sql = re.sub(r'SET search_path.*?;', '', sql)
sql = re.sub(r'CREATE EXTENSION.*?;', '', sql)
sql = re.sub(r'COMMENT ON.*?;', '', sql, flags=re.DOTALL)
sql = re.sub(r'CREATE INDEX.*?;', '', sql)
sql = re.sub(r'ALTER TABLE.*?;', '', sql, flags=re.DOTALL)

# ── Bỏ TRIGGER + FUNCTION (không dùng được trong SQLite) ─────
sql = re.sub(r'CREATE OR REPLACE FUNCTION.*?\$\$;', '', sql, flags=re.DOTALL)
sql = re.sub(r'CREATE TRIGGER.*?;', '', sql, flags=re.DOTALL)
sql = re.sub(r'RETURN NEW.*?END.*?\$\$', '', sql, flags=re.DOTALL)

# ── Type conversion ───────────────────────────────────────────
sql = re.sub(r'TIMESTAMPTZ', 'TEXT', sql)
sql = re.sub(r'\bSERIAL\b', 'INTEGER', sql)
sql = re.sub(r'\bBOOLEAN\b', 'INTEGER', sql)
sql = re.sub(r'NUMERIC\(\d+,\s*\d+\)', 'REAL', sql)
sql = re.sub(r'VARCHAR\(\d+\)', 'TEXT', sql)
sql = re.sub(r'\bDATE\b', 'TEXT', sql)
sql = re.sub(r'DEFAULT NOW\(\)', "DEFAULT (datetime('now'))", sql)

# ── Bỏ FK + UNIQUE constraints (gây lỗi parse) ───────────────
sql = re.sub(r'REFERENCES\s+\w+\s*\([^)]+\)', '', sql)
sql = re.sub(r',\s*UNIQUE\s*\([^)]+\)', '', sql)
sql = re.sub(r'ON DELETE\s+\w+', '', sql)
sql = re.sub(r'ON UPDATE\s+\w+', '', sql)

# ── Bỏ comment tiếng Việt gây lỗi parse ─────────────────────
sql = re.sub(r'--[^\n]*', '', sql)

# ── Tạo DB ────────────────────────────────────────────────────
import os
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
statements = [s.strip() for s in sql.split(';') if s.strip() and 'CREATE TABLE' in s]

ok, fail = 0, 0
for stmt in statements:
    try:
        conn.execute(stmt)
        ok += 1
    except Exception as e:
        print(f"❌ {e}\n   {stmt[:150]}\n")
        fail += 1

conn.commit()

cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [r[0] for r in cur.fetchall()]
print(f"\n✅ Tạo OK: {ok} | ❌ Fail: {fail}")
print(f"📋 Bảng: {tables}")
conn.close()

❌ near "FROM": syntax error
   CREATE TABLE sales_order (
    order_id            INTEGER          PRIMARY KEY,
    so_number           TEXT     NOT NULL UNIQUE,  
    invoice_symbo


✅ Tạo OK: 8 | ❌ Fail: 1
📋 Bảng: ['customer', 'fact_sales', 'order_line', 'product', 'product_group', 'product_line', 'product_price', 'province']


In [16]:
# Cell 16d — Fix sales_order riêng
conn = sqlite3.connect(db_path)

# Tạo sales_order thủ công theo schema
conn.execute("""
CREATE TABLE IF NOT EXISTS sales_order (
    order_id            INTEGER     PRIMARY KEY,
    so_number           TEXT        NOT NULL UNIQUE,
    invoice_symbol      TEXT,
    invoice_number      TEXT,
    order_date          TEXT        NOT NULL,
    customer_code       TEXT        NOT NULL,
    warehouse_code      TEXT,
    payment_method      TEXT,
    notes               TEXT,
    total_amount        REAL        DEFAULT 0,
    total_quantity      INTEGER     DEFAULT 0,
    processing_status   TEXT        DEFAULT 'pending',
    created_at          TEXT        DEFAULT (datetime('now')),
    updated_at          TEXT        DEFAULT (datetime('now'))
)
""")
conn.commit()

# Kiểm tra tất cả bảng
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [r[0] for r in cur.fetchall()]
print(f"✅ Đủ {len(tables)} bảng:", tables)
conn.close()

✅ Đủ 9 bảng: ['customer', 'fact_sales', 'order_line', 'product', 'product_group', 'product_line', 'product_price', 'province', 'sales_order']


In [17]:
# Cell 17 — Import data gốc từ 02_import_data.sql
conn = sqlite3.connect(db_path)

with open("/tmp/tnbike_db/02_import_data.sql", "r", encoding="utf-8") as f:
    sql_data = f.read()

# Adapt PostgreSQL → SQLite
sql_data = re.sub(r'SET search_path.*?;', '', sql_data)
sql_data = re.sub(r'INSERT INTO tnbike\.', 'INSERT INTO ', sql_data)
sql_data = re.sub(r'\bTRUE\b', '1', sql_data)
sql_data = re.sub(r'\bFALSE\b', '0', sql_data)
sql_data = re.sub(r'NOW\(\)', "datetime('now')", sql_data)
sql_data = re.sub(r'--[^\n]*', '', sql_data)

conn.executescript(sql_data)
conn.commit()

cur = conn.cursor()
for t in ["product_group","product","customer","sales_order","order_line","fact_sales"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:20s}: {cur.fetchone()[0]:,} rows")

conn.close()

OperationalError: near "SET": syntax error

In [18]:
# Cell 17b — Debug import data
conn = sqlite3.connect(db_path)

with open("/tmp/tnbike_db/02_import_data.sql", "r", encoding="utf-8") as f:
    sql_data = f.read()

# Adapt
sql_data = re.sub(r'--[^\n]*', '', sql_data)
sql_data = re.sub(r'SET\s+\w+.*?;', '', sql_data)          # bỏ mọi SET ...;
sql_data = re.sub(r'INSERT INTO tnbike\.', 'INSERT INTO ', sql_data)
sql_data = re.sub(r'\bTRUE\b', '1', sql_data)
sql_data = re.sub(r'\bFALSE\b', '0', sql_data)
sql_data = re.sub(r'NOW\(\)', "datetime('now')", sql_data)
sql_data = re.sub(r'BEGIN;', '', sql_data)
sql_data = re.sub(r'COMMIT;', '', sql_data)

# Chạy từng statement
ok, fail = 0, 0
statements = [s.strip() for s in sql_data.split(';') if s.strip()]
for i, stmt in enumerate(statements):
    try:
        conn.execute(stmt)
        ok += 1
    except Exception as e:
        fail += 1
        if fail <= 3:  # chỉ in 3 lỗi đầu
            print(f"❌ #{i}: {e}")
            print(f"   {stmt[:200]}\n")

conn.commit()
print(f"\n✅ OK: {ok:,} | ❌ Fail: {fail}")

cur = conn.cursor()
for t in ["product_group","product","customer","sales_order","order_line","fact_sales"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:20s}: {cur.fetchone()[0]:,} rows")

conn.close()

❌ #5: unrecognized token: "'SN"
   INSERT INTO customer (customer_code, customer_name, tax_code, address, province_id) VALUES
  ('KH-00001', 'HỘ KINH DOANH TÚ ANH', '257951869', 'Phường Thanh Lương, Quận Hai Bà Trưng, Hà Nội', (SELECT 

❌ #6: near "231": syntax error
   231 tổ 32 khu 4A, Phường Cửa Ông, Quảng Ninh', (SELECT province_id FROM province WHERE province_name = 'Quảng Ninh' LIMIT 1)),
  ('KH-00624', 'CÔNG TY TNHH THƯƠNG MẠI HÙNG CƯỜNG', '191638308', 'Khu Lạ

❌ #9: near "FROM": syntax error
   INSERT INTO fact_sales (
    order_date, fiscal_year, fiscal_quarter, fiscal_month, week_of_year,
    so_number, order_id, line_id,
    customer_code, customer_name, province_id, province_name, region


✅ OK: 8 | ❌ Fail: 3
  product_group       : 5 rows
  product             : 247 rows
  customer            : 0 rows
  sales_order         : 1,627 rows
  order_line          : 17,031 rows
  fact_sales          : 0 rows


In [19]:
# Cell 17c — Import đúng cách với multiline INSERT

import sqlite3, re

conn = sqlite3.connect(db_path)

with open("/tmp/tnbike_db/02_import_data.sql", "r", encoding="utf-8") as f:
    sql_data = f.read()

# Adapt
sql_data = re.sub(r'--[^\n]*', '', sql_data)
sql_data = re.sub(r'SET\s+\S+.*?;', '', sql_data, flags=re.DOTALL)
sql_data = re.sub(r'INSERT INTO tnbike\.', 'INSERT INTO ', sql_data)
sql_data = re.sub(r'\bTRUE\b', '1', sql_data)
sql_data = re.sub(r'\bFALSE\b', '0', sql_data)
sql_data = re.sub(r'NOW\(\)', "datetime('now')", sql_data)

# Dùng executescript thay vì split thủ công
# Bọc trong transaction
try:
    conn.executescript("BEGIN;\n" + sql_data + "\nCOMMIT;")
except Exception as e:
    print(f"❌ executescript lỗi: {e}")
    # Thử từng block INSERT lớn
    blocks = re.split(r'\n(?=INSERT INTO)', sql_data)
    ok = fail = 0
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        try:
            conn.executescript(block)
            ok += 1
        except Exception as e2:
            fail += 1
            if fail <= 2:
                print(f"❌ Block fail: {e2}")
                print(f"   {block[:150]}\n")
    print(f"Blocks: ✅{ok} ❌{fail}")

conn.commit()

cur = conn.cursor()
for t in ["product_group","product","customer","sales_order","order_line","fact_sales"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:20s}: {cur.fetchone()[0]:,} rows")

conn.close()

❌ executescript lỗi: cannot start a transaction within a transaction
❌ Block fail: UNIQUE constraint failed: product_group.group_code
   INSERT INTO product_group (group_code, group_name, description) VALUES
  ('CITYBIKE_P', 'Xe phổ thông', 'Xe đạp phổ thông dành cho người lớn: các dòng

❌ Block fail: UNIQUE constraint failed: product.product_code
   INSERT INTO product (product_code, product_name, line_id, color, unit) VALUES
  ('000214004000000', 'Xe đạp Thống Nhất GN 05-26', (SELECT line_id FROM

Blocks: ✅5 ❌5
  product_group       : 5 rows
  product             : 247 rows
  customer            : 702 rows
  sales_order         : 1,627 rows
  order_line          : 34,062 rows
  fact_sales          : 0 rows


In [20]:
# Cell 17d — Kiểm tra data hiện có
conn = sqlite3.connect(db_path)
cur = conn.cursor()

for t in ["product_group","product","customer","sales_order","order_line","fact_sales","province"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:20s}: {cur.fetchone()[0]:,} rows")

conn.close()

  product_group       : 5 rows
  product             : 247 rows
  customer            : 702 rows
  sales_order         : 1,627 rows
  order_line          : 34,062 rows
  fact_sales          : 0 rows
  province            : 75 rows


In [21]:
# Cell 18 — Nếu đã có data → bắt đầu INSERT email tháng 3 luôn!
# Xem schema sales_order và order_line thực tế
conn = sqlite3.connect(db_path)
cur = conn.cursor()

for t in ["sales_order", "order_line", "fact_sales"]:
    cur.execute(f"PRAGMA table_info({t})")
    cols = cur.fetchall()
    print(f"\n── {t} ──")
    for c in cols:
        print(f"   {c[1]:30s} {c[2]}")

# Xem 1 row mẫu
print("\n── sales_order mẫu ──")
cur.execute("SELECT * FROM sales_order LIMIT 2")
for row in cur.fetchall():
    print(row)

conn.close()


── sales_order ──
   order_id                       INTEGER
   so_number                      TEXT
   invoice_symbol                 TEXT
   invoice_number                 TEXT
   order_date                     TEXT
   customer_code                  TEXT
   warehouse_code                 TEXT
   payment_method                 TEXT
   notes                          TEXT
   total_amount                   REAL
   total_quantity                 INTEGER
   processing_status              TEXT
   created_at                     TEXT
   updated_at                     TEXT

── order_line ──
   line_id                        INTEGER
   order_id                       INTEGER
   so_number                      TEXT
   product_code                   TEXT
   quantity                       REAL
   unit_price                     REAL
   line_total                     REAL
   created_at                     TEXT

── fact_sales ──
   fact_id                        BIGSERIAL
   order_date                  

In [22]:
# Cell 19 — Xem max order_id và so_number hiện tại
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("SELECT MAX(order_id) FROM sales_order")
max_order_id = cur.fetchone()[0]

cur.execute("SELECT MAX(line_id) FROM order_line")
max_line_id = cur.fetchone()[0]

cur.execute("SELECT MAX(fact_id) FROM fact_sales")
max_fact_id = cur.fetchone()[0]

print(f"Max order_id : {max_order_id}")
print(f"Max line_id  : {max_line_id}")
print(f"Max fact_id  : {max_fact_id}")

# Xem invoice_symbol pattern
cur.execute("SELECT DISTINCT invoice_symbol FROM sales_order LIMIT 5")
print(f"Invoice symbols: {cur.fetchall()}")

# Xem customer mẫu để biết cách map
cur.execute("SELECT customer_code, customer_name FROM customer LIMIT 3")
print(f"Customer mẫu: {cur.fetchall()}")

conn.close()

Max order_id : 1627
Max line_id  : 34062
Max fact_id  : None
Invoice symbols: [('C25TTN',), ('C26TTN',)]
Customer mẫu: [('KH-00001', 'HỘ KINH DOANH TÚ ANH'), ('KH-00002', 'CÔNG TY TNHH THƯƠNG MẠI VIỆT ANH'), ('KH-00003', 'CÔNG TY CỔ PHẦN THƯƠNG MẠI NAM TIẾN')]


In [23]:
# Cell 21 — Map customer_name → customer_code + INSERT sales_order, email_log
import sqlite3, pandas as pd
from datetime import datetime

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Load customer lookup
cur.execute("SELECT customer_code, customer_name FROM customer")
cust_map = {name.upper().strip(): code for code, name in cur.fetchall()}

# Tạo bảng email_log nếu chưa có
conn.execute("""
CREATE TABLE IF NOT EXISTS email_log (
    log_id             INTEGER PRIMARY KEY,
    message_id         TEXT,
    from_address       TEXT,
    received_at        TEXT,
    attachment_name    TEXT,
    processing_status  TEXT DEFAULT 'processed',
    created_at         TEXT DEFAULT (datetime('now'))
)
""")

start_order_id = 1628
start_line_id  = 34063
invoice_symbol = 'C26TTN'

orders_inserted = 0
logs_inserted   = 0
not_found       = []

for i, row in df.iterrows():
    cust_code = cust_map.get(str(row["dai_ly"]).upper().strip())

    if not cust_code:
        not_found.append(row["dai_ly"])
        cust_code = "KH-UNKNOWN"

    order_id  = start_order_id + i
    so_number = row["ma_don"].replace(".", "")  # BH26.0935 → BH260935
    so_number = f"BH26.{row['ma_don'].split('.')[-1]}"  # chuẩn hóa
    order_date = row["ngay"].strftime("%Y-%m-%d")

    # Invoice number = số thứ tự trong tháng
    invoice_number = str(i + 1)

    # INSERT sales_order
    cur.execute("""
        INSERT OR IGNORE INTO sales_order
        (order_id, so_number, invoice_symbol, invoice_number, order_date,
         customer_code, total_amount, total_quantity, processing_status)
        VALUES (?,?,?,?,?,?,?,?,?)
    """, (order_id, so_number, invoice_symbol, invoice_number, order_date,
          cust_code, float(row["tri_gia"]), int(row["so_chiec"]), "processed"))

    # INSERT email_log
    cur.execute("""
        INSERT INTO email_log
        (message_id, from_address, received_at, attachment_name, processing_status)
        VALUES (?,?,?,?,?)
    """, (so_number, f"dathang@email.vn", order_date, f"{row['file']}", "processed"))

    orders_inserted += 1
    logs_inserted   += 1

conn.commit()

print(f"✅ sales_order inserted : {orders_inserted}")
print(f"✅ email_log inserted   : {logs_inserted}")
print(f"⚠️  Customer not found  : {len(not_found)} → {list(set(not_found))[:5]}")

conn.close()

✅ sales_order inserted : 1132
✅ email_log inserted   : 1132
⚠️  Customer not found  : 4 → ['CÔNG TY TNHH THƯƠNG MẠI TÂN TIẾN', 'CỬA HÀNG XE ĐẠP SƠN HÀ', 'CỬA HÀNG XE ĐẠP TRÍ ĐỨC']


In [24]:
# Cell 22 — INSERT order_line
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("SELECT order_id, so_number, total_amount, total_quantity FROM sales_order WHERE invoice_symbol='C26TTN'")
orders = cur.fetchall()

line_id = 34063
for order_id, so_number, total_amount, total_quantity in orders:
    qty = total_quantity if total_quantity and total_quantity > 0 else 1
    cur.execute("""
        INSERT OR IGNORE INTO order_line
        (line_id, order_id, so_number, product_code, quantity, unit_price, line_total)
        VALUES (?,?,?,?,?,?,?)
    """, (line_id, order_id, so_number, 'UNKNOWN',
          qty,
          round(total_amount / qty, 2),
          total_amount))
    line_id += 1

conn.commit()
print(f"✅ order_line inserted: {line_id - 34063} rows")
conn.close()

✅ order_line inserted: 2066 rows


In [25]:
# Cell 23 — Cập nhật fact_sales tháng 3/2026
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("""
INSERT INTO fact_sales
    (order_date, fiscal_year, fiscal_quarter, fiscal_month, week_of_year,
     so_number, order_id, line_id, customer_code, customer_name,
     province_id, province_name, region,
     product_code, quantity, unit_price, line_total)
SELECT
    so.order_date,
    CAST(strftime('%Y', so.order_date) AS INTEGER),
    CAST((CAST(strftime('%m', so.order_date) AS INTEGER) + 2) / 3 AS INTEGER),
    CAST(strftime('%m', so.order_date) AS INTEGER),
    CAST(strftime('%W', so.order_date) AS INTEGER),
    so.so_number,
    so.order_id,
    ol.line_id,
    so.customer_code,
    COALESCE(c.customer_name, so.customer_code),
    c.province_id,
    p.province_name,
    p.region,
    ol.product_code,
    ol.quantity,
    ol.unit_price,
    ol.line_total
FROM sales_order so
JOIN order_line ol  ON ol.order_id   = so.order_id
LEFT JOIN customer c  ON c.customer_code = so.customer_code
LEFT JOIN province p  ON p.province_id   = c.province_id
WHERE so.invoice_symbol = 'C26TTN'
""")

conn.commit()
print(f"✅ fact_sales inserted: {cur.rowcount:,} rows")

# Kiểm tra tổng quan
cur.execute("""
    SELECT fiscal_month, COUNT(*) as don_hang, SUM(line_total) as doanh_thu
    FROM fact_sales
    WHERE fiscal_year = 2026
    GROUP BY fiscal_month
    ORDER BY fiscal_month
""")
print("\n📊 fact_sales 2026:")
print(f"{'Tháng':>8} {'Đơn hàng':>12} {'Doanh thu':>20}")
for row in cur.fetchall():
    print(f"  T{row[0]:02d}   {row[1]:>10,}   {row[2]:>18,.0f}")

conn.close()

✅ fact_sales inserted: 21,298 rows

📊 fact_sales 2026:
   Tháng     Đơn hàng            Doanh thu
  T01       10,038       42,270,147,220
  T02       10,128       38,777,738,266
  T03        1,132       40,804,047,133


In [26]:
# Cell 24 — Báo cáo tổng kết pipeline
conn = sqlite3.connect(db_path)
cur = conn.cursor()

print("=" * 55)
print("  TỔNG KẾT PIPELINE EMAIL → DATABASE")
print("=" * 55)

# Bảng chính
for t, label in [
    ("email_log",   "Email đã xử lý"),
    ("sales_order", "Đơn hàng tháng 3"),
    ("order_line",  "Order lines tháng 3"),
    ("fact_sales",  "Fact sales tháng 3"),
]:
    if t == "sales_order":
        cur.execute(f"SELECT COUNT(*) FROM {t} WHERE invoice_symbol='C26TTN'")
    elif t in ("order_line", "fact_sales"):
        cur.execute(f"SELECT COUNT(*) FROM {t} WHERE so_number LIKE 'BH26.%'")
    else:
        cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {label:25s}: {cur.fetchone()[0]:>6,} rows")

print()

# Doanh thu theo tháng
cur.execute("""
    SELECT fiscal_year, fiscal_month,
           COUNT(DISTINCT so_number) don_hang,
           SUM(line_total) doanh_thu
    FROM fact_sales
    GROUP BY fiscal_year, fiscal_month
    ORDER BY fiscal_year, fiscal_month
""")
print(f"  {'Năm-Tháng':12} {'Đơn':>8} {'Doanh thu (tỷ)':>16}")
print("  " + "-" * 40)
for y, m, d, r in cur.fetchall():
    print(f"  {y}-T{m:02d}      {d:>8,}   {r/1e9:>14.2f}")

print()
print(f"  ⚠️  Lưu ý: T03/2026 mỗi đơn chỉ có 1 order_line tổng hợp")
print(f"     (email không chứa chi tiết SKU — cần PDF để bổ sung)")
print("=" * 55)

conn.close()

  TỔNG KẾT PIPELINE EMAIL → DATABASE
  Email đã xử lý           :  1,132 rows
  Đơn hàng tháng 3         :  2,066 rows
  Order lines tháng 3      : 21,298 rows
  Fact sales tháng 3       : 21,298 rows

  Năm-Tháng         Đơn   Doanh thu (tỷ)
  ----------------------------------------
  2026-T01           482            42.27
  2026-T02           452            38.78
  2026-T03         1,132            40.80

  ⚠️  Lưu ý: T03/2026 mỗi đơn chỉ có 1 order_line tổng hợp
     (email không chứa chi tiết SKU — cần PDF để bổ sung)


In [27]:
# Cell 25 — Kiểm tra và fix duplicate trong sales_order
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Xem tại sao có 2,066 thay vì 1,132
cur.execute("""
    SELECT so_number, COUNT(*) as cnt
    FROM sales_order
    WHERE invoice_symbol = 'C26TTN'
    GROUP BY so_number
    HAVING cnt > 1
    LIMIT 5
""")
dupes = cur.fetchall()
print(f"Duplicate so_number: {len(dupes)} cases")
for d in dupes[:5]:
    print(f"  {d}")

# Xem range order_id tháng 3
cur.execute("""
    SELECT MIN(order_id), MAX(order_id), COUNT(*)
    FROM sales_order WHERE invoice_symbol='C26TTN'
""")
print("\nRange order_id C26TTN:", cur.fetchone())

conn.close()

Duplicate so_number: 0 cases

Range order_id C26TTN: (694, 2759, 2066)


In [28]:
# Cell 26 — Nếu có duplicate → xóa và insert lại đúng
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Xóa các order trùng (giữ order_id nhỏ nhất)
cur.execute("""
    DELETE FROM sales_order
    WHERE order_id NOT IN (
        SELECT MIN(order_id)
        FROM sales_order
        WHERE invoice_symbol = 'C26TTN'
        GROUP BY so_number
    )
    AND invoice_symbol = 'C26TTN'
""")
deleted = cur.rowcount

# Xóa fact_sales và order_line cũ của T03 để rebuild sạch
cur.execute("DELETE FROM fact_sales WHERE fiscal_year=2026 AND fiscal_month=3")
cur.execute("DELETE FROM order_line WHERE so_number LIKE 'BH26.%' AND line_id >= 34063")

conn.commit()
print(f"🗑️  Đã xóa {deleted} duplicate sales_order")

cur.execute("SELECT COUNT(*) FROM sales_order WHERE invoice_symbol='C26TTN'")
print(f"✅ sales_order T03 sau fix: {cur.fetchone()[0]:,} rows")

conn.close()

🗑️  Đã xóa 0 duplicate sales_order
✅ sales_order T03 sau fix: 2,066 rows


In [29]:
# Cell 25b — Điều tra overlap
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Xem order_id 694 là đơn gì
cur.execute("SELECT order_id, so_number, invoice_symbol, order_date FROM sales_order WHERE order_id IN (694, 695, 696)")
print("order_id 694-696:")
for r in cur.fetchall(): print(" ", r)

# Xem phân bố invoice_symbol
cur.execute("SELECT invoice_symbol, COUNT(*), MIN(order_id), MAX(order_id) FROM sales_order GROUP BY invoice_symbol")
print("\nPhân bố theo invoice_symbol:")
for r in cur.fetchall(): print(" ", r)

# Xem có bao nhiêu BH26 trong so_number
cur.execute("SELECT COUNT(*) FROM sales_order WHERE so_number LIKE 'BH26%'")
print("\nĐơn BH26:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM sales_order WHERE so_number LIKE 'BH25%'")
print("Đơn BH25:", cur.fetchone()[0])

conn.close()

order_id 694-696:
  (694, 'BH26.0001', 'C26TTN', '2026-01-04')
  (695, 'BH26.0005', 'C26TTN', '2026-01-06')
  (696, 'BH26.0004', 'C26TTN', '2026-01-06')

Phân bố theo invoice_symbol:
  ('C25TTN', 693, 1, 693)
  ('C26TTN', 2066, 694, 2759)

Đơn BH26: 2066
Đơn BH25: 693


In [30]:
# Cell 26 — Verify toàn bộ pipeline FINAL
conn = sqlite3.connect(db_path)
cur = conn.cursor()

print("=" * 60)
print("  ✅ PIPELINE HOÀN CHỈNH — VERIFY CUỐI")
print("=" * 60)

# Đếm theo tháng
cur.execute("""
    SELECT
        strftime('%Y-%m', order_date) as thang,
        invoice_symbol,
        COUNT(*) as don_hang,
        SUM(total_amount) as doanh_thu
    FROM sales_order
    GROUP BY thang, invoice_symbol
    ORDER BY thang
""")
print(f"\n{'Tháng':>10} {'Symbol':>8} {'Đơn':>8} {'Doanh thu (tỷ)':>16}")
print("  " + "-"*46)
for thang, sym, don, dt in cur.fetchall():
    print(f"  {thang}   {sym}  {don:>8,}   {dt/1e9:>14.2f}")

# Verify 4 bảng đã ghi
print(f"\n{'Bảng':>20} {'Rows':>10}")
print("  " + "-"*32)
for t in ["email_log", "sales_order", "order_line", "fact_sales"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:>20}: {cur.fetchone()[0]:>10,}")

print("\n  ⚠️  order_line T03: 1 dòng tổng hợp/đơn (email không có SKU)")
print("=" * 60)
conn.close()

  ✅ PIPELINE HOÀN CHỈNH — VERIFY CUỐI

     Tháng   Symbol      Đơn   Doanh thu (tỷ)
  ----------------------------------------------
  2025-01   C25TTN        61             0.00
  2025-02   C25TTN       185             0.00
  2025-03   C25TTN       447             0.00
  2026-01   C26TTN       482             0.00
  2026-02   C26TTN       452             0.00
  2026-03   C26TTN     1,132            40.80

                Bảng       Rows
  --------------------------------
             email_log:      1,132
           sales_order:      2,759
            order_line:     34,062
            fact_sales:     20,166

  ⚠️  order_line T03: 1 dòng tổng hợp/đơn (email không có SKU)


In [31]:
# Cell 27 — Fix total_amount từ order_line + rebuild fact_sales đầy đủ
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Fix 1: Update total_amount, total_quantity trong sales_order từ order_line
cur.execute("""
    UPDATE sales_order
    SET total_amount   = (SELECT COALESCE(SUM(line_total), 0) FROM order_line WHERE order_line.order_id = sales_order.order_id),
        total_quantity = (SELECT COALESCE(SUM(quantity), 0)   FROM order_line WHERE order_line.order_id = sales_order.order_id)
""")
print(f"✅ Updated total_amount/quantity: {cur.rowcount:,} orders")

conn.commit()

# Verify
cur.execute("""
    SELECT strftime('%Y-%m', order_date) as thang,
           COUNT(*) as don, SUM(total_amount) as dt
    FROM sales_order
    GROUP BY thang ORDER BY thang
""")
print(f"\n{'Tháng':>10} {'Đơn':>8} {'Doanh thu (tỷ)':>16}")
print("  " + "-"*36)
for thang, don, dt in cur.fetchall():
    print(f"  {thang}   {don:>6,}   {dt/1e9 if dt else 0:>14.2f}")

conn.close()

✅ Updated total_amount/quantity: 2,759 orders

     Tháng      Đơn   Doanh thu (tỷ)
  ------------------------------------
  2025-01       61             6.40
  2025-02      185            12.67
  2025-03      447            37.16
  2026-01      482            42.27
  2026-02      452            38.78
  2026-03    1,132             0.00


In [32]:
# Cell 28 — Rebuild fact_sales đầy đủ (xóa sạch rồi insert lại)
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("DELETE FROM fact_sales")
print(f"🗑️  Cleared fact_sales")

cur.execute("""
INSERT INTO fact_sales
    (order_date, fiscal_year, fiscal_quarter, fiscal_month, week_of_year,
     so_number, order_id, line_id, customer_code, customer_name,
     province_id, province_name, region,
     product_code, quantity, unit_price, line_total)
SELECT
    so.order_date,
    CAST(strftime('%Y', so.order_date) AS INTEGER),
    CAST((CAST(strftime('%m', so.order_date) AS INTEGER) + 2) / 3 AS INTEGER),
    CAST(strftime('%m', so.order_date) AS INTEGER),
    CAST(strftime('%W', so.order_date) AS INTEGER),
    so.so_number, so.order_id, ol.line_id,
    so.customer_code,
    COALESCE(c.customer_name, so.customer_code),
    c.province_id, p.province_name, p.region,
    ol.product_code, ol.quantity, ol.unit_price, ol.line_total
FROM sales_order so
JOIN order_line ol ON ol.order_id = so.order_id
LEFT JOIN customer c ON c.customer_code = so.customer_code
LEFT JOIN province p ON p.province_id = c.province_id
""")
conn.commit()
print(f"✅ fact_sales rebuilt: {cur.rowcount:,} rows")

# Summary cuối
cur.execute("""
    SELECT fiscal_year, fiscal_month,
           COUNT(DISTINCT so_number), SUM(line_total)
    FROM fact_sales GROUP BY fiscal_year, fiscal_month
    ORDER BY fiscal_year, fiscal_month
""")
print(f"\n{'Năm-T':>8} {'Đơn':>8} {'Doanh thu (tỷ)':>16}")
print("  "+"-"*34)
for y,m,d,r in cur.fetchall():
    print(f"  {y}-T{m:02d}  {d:>6,}   {r/1e9:>14.2f}")

conn.close()

🗑️  Cleared fact_sales
✅ fact_sales rebuilt: 34,062 rows

   Năm-T      Đơn   Doanh thu (tỷ)
  ----------------------------------
  2025-T01      61             6.40
  2025-T02     185            12.67
  2025-T03     447            37.16
  2026-T01     482            42.27
  2026-T02     452            38.78


In [33]:
# Cell 29 — Fix: update total_amount T03 + re-insert fact_sales T03
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Kiểm tra order_line T03 có data không
cur.execute("""
    SELECT COUNT(*), SUM(line_total) FROM order_line ol
    JOIN sales_order so ON so.order_id = ol.order_id
    WHERE so.invoice_symbol = 'C26TTN'
    AND so.order_date LIKE '2026-03%'
""")
cnt, total = cur.fetchone()
print(f"order_line T03: {cnt:,} rows | total: {total:,.0f}" if total else f"order_line T03: {cnt} rows | EMPTY!")

# Xem order_line có line_id >= 34063 không
cur.execute("SELECT COUNT(*), MIN(line_id), MAX(line_id) FROM order_line WHERE line_id >= 34063")
print("order_line >= 34063:", cur.fetchone())

conn.close()

order_line T03: 0 rows | EMPTY!
order_line >= 34063: (0, None, None)


In [34]:
# Cell 30 — Re-insert order_line T03 nếu bị mất
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Lấy orders T03 chưa có order_line
cur.execute("""
    SELECT so.order_id, so.so_number, so.total_amount, so.total_quantity
    FROM sales_order so
    LEFT JOIN order_line ol ON ol.order_id = so.order_id
    WHERE so.order_date LIKE '2026-03%'
    AND ol.line_id IS NULL
""")
missing = cur.fetchall()
print(f"Orders T03 chưa có order_line: {len(missing)}")

# Insert order_line cho T03
cur.execute("SELECT MAX(line_id) FROM order_line")
line_id = (cur.fetchone()[0] or 0) + 1

for order_id, so_number, total_amount, total_quantity in missing:
    qty = total_quantity if total_quantity and total_quantity > 0 else 1
    cur.execute("""
        INSERT INTO order_line (line_id, order_id, so_number, product_code, quantity, unit_price, line_total)
        VALUES (?,?,?,?,?,?,?)
    """, (line_id, order_id, so_number, 'UNKNOWN', qty,
          round(total_amount/qty, 2) if qty else 0, total_amount))
    line_id += 1

conn.commit()
print(f"✅ Inserted {len(missing)} order_lines cho T03")

# Update total_amount T03 từ email df
cur.execute("""
    UPDATE sales_order SET
        total_amount   = (SELECT COALESCE(SUM(line_total),0) FROM order_line WHERE order_line.order_id = sales_order.order_id),
        total_quantity = (SELECT COALESCE(SUM(quantity),0)   FROM order_line WHERE order_line.order_id = sales_order.order_id)
    WHERE order_date LIKE '2026-03%'
""")
conn.commit()
print(f"✅ Updated total_amount T03")

# Append fact_sales T03
cur.execute("""
INSERT INTO fact_sales
    (order_date, fiscal_year, fiscal_quarter, fiscal_month, week_of_year,
     so_number, order_id, line_id, customer_code, customer_name,
     province_id, province_name, region,
     product_code, quantity, unit_price, line_total)
SELECT
    so.order_date,
    2026, 1, 3,
    CAST(strftime('%W', so.order_date) AS INTEGER),
    so.so_number, so.order_id, ol.line_id,
    so.customer_code,
    COALESCE(c.customer_name, so.customer_code),
    c.province_id, p.province_name, p.region,
    ol.product_code, ol.quantity, ol.unit_price, ol.line_total
FROM sales_order so
JOIN order_line ol ON ol.order_id = so.order_id
LEFT JOIN customer c ON c.customer_code = so.customer_code
LEFT JOIN province p ON p.province_id = c.province_id
WHERE so.order_date LIKE '2026-03%'
""")
conn.commit()
print(f"✅ fact_sales T03 inserted: {cur.rowcount:,} rows")

# Final summary
cur.execute("""
    SELECT fiscal_year, fiscal_month, COUNT(DISTINCT so_number), SUM(line_total)
    FROM fact_sales GROUP BY fiscal_year, fiscal_month ORDER BY 1,2
""")
print(f"\n{'Năm-T':>8} {'Đơn':>8} {'Doanh thu (tỷ)':>16}")
print("  "+"-"*36)
for y,m,d,r in cur.fetchall():
    print(f"  {y}-T{m:02d}  {d:>6,}   {r/1e9:>14.2f}")

conn.close()

Orders T03 chưa có order_line: 1132
✅ Inserted 1132 order_lines cho T03
✅ Updated total_amount T03
✅ fact_sales T03 inserted: 1,132 rows

   Năm-T      Đơn   Doanh thu (tỷ)
  ------------------------------------
  2025-T01      61             6.40
  2025-T02     185            12.67
  2025-T03     447            37.16
  2026-T01     482            42.27
  2026-T02     452            38.78
  2026-T03   1,132             0.00


In [35]:
# Cell 31 — Update total_amount trực tiếp từ df email
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Xem df có cột gì
print("df columns:", df.columns.tolist())
print(df[["ma_don","tri_gia","so_chiec"]].head(3))

df columns: ['file', 'ma_don', 'ngay', 'dai_ly', 'so_chiec', 'tri_gia']
      ma_don   tri_gia  so_chiec
0  BH26.1932  48565357        31
1  BH26.1865  45345079        29
2  BH26.1014  62442770        35


In [36]:
# Cell 32 — Map df → sales_order theo so_number, update amount + line_total
conn = sqlite3.connect(db_path)
cur = conn.cursor()

updated = 0
for _, row in df.iterrows():
    so_number = f"BH26.{str(row['ma_don']).split('.')[-1]}"
    tri_gia   = float(row["tri_gia"])
    so_chiec  = int(row["so_chiec"]) if row["so_chiec"] else 1

    # Update sales_order
    cur.execute("""
        UPDATE sales_order
        SET total_amount=?, total_quantity=?
        WHERE so_number=? AND order_date LIKE '2026-03%'
    """, (tri_gia, so_chiec, so_number))

    # Update order_line
    cur.execute("""
        UPDATE order_line
        SET line_total=?, quantity=?, unit_price=?
        WHERE so_number=?
    """, (tri_gia, so_chiec, round(tri_gia/so_chiec,2) if so_chiec else 0, so_number))

    if cur.rowcount: updated += 1

conn.commit()
print(f"✅ Updated: {updated:,} orders")

# Update fact_sales T03
cur.execute("""
    UPDATE fact_sales SET
        line_total = (SELECT ol.line_total FROM order_line ol WHERE ol.line_id = fact_sales.line_id),
        quantity   = (SELECT ol.quantity   FROM order_line ol WHERE ol.line_id = fact_sales.line_id),
        unit_price = (SELECT ol.unit_price FROM order_line ol WHERE ol.line_id = fact_sales.line_id)
    WHERE fiscal_year=2026 AND fiscal_month=3
""")
conn.commit()
print(f"✅ fact_sales T03 updated: {cur.rowcount:,} rows")

# Final check
cur.execute("""
    SELECT fiscal_year, fiscal_month, COUNT(DISTINCT so_number), SUM(line_total)
    FROM fact_sales GROUP BY fiscal_year, fiscal_month ORDER BY 1,2
""")
print(f"\n{'Năm-T':>8} {'Đơn':>8} {'Doanh thu (tỷ)':>16}")
print("  "+"-"*36)
for y,m,d,r in cur.fetchall():
    print(f"  {y}-T{m:02d}  {d:>6,}   {r/1e9 if r else 0:>14.2f}")

conn.close()

✅ Updated: 1,132 orders
✅ fact_sales T03 updated: 1,132 rows

   Năm-T      Đơn   Doanh thu (tỷ)
  ------------------------------------
  2025-T01      61             6.40
  2025-T02     185            12.67
  2025-T03     447            37.16
  2026-T01     482            42.27
  2026-T02     452            38.78
  2026-T03   1,132            40.80


In [37]:
# Cell 32 — Update amount trực tiếp từ df
conn = sqlite3.connect(db_path)
cur = conn.cursor()

updated_so = 0
updated_ol = 0

for _, row in df.iterrows():
    so_number = str(row["ma_don"]).strip()   # đã là BH26.1932
    tri_gia   = float(row["tri_gia"])
    so_chiec  = int(row["so_chiec"]) if row["so_chiec"] and row["so_chiec"] > 0 else 1

    cur.execute("""
        UPDATE sales_order
        SET total_amount=?, total_quantity=?
        WHERE so_number=?
    """, (tri_gia, so_chiec, so_number))
    updated_so += cur.rowcount

    cur.execute("""
        UPDATE order_line
        SET line_total=?, quantity=?, unit_price=?
        WHERE so_number=?
    """, (tri_gia, so_chiec, round(tri_gia/so_chiec, 2), so_number))
    updated_ol += cur.rowcount

conn.commit()
print(f"✅ sales_order updated : {updated_so:,}")
print(f"✅ order_line updated  : {updated_ol:,}")

# Update fact_sales T03 từ order_line
cur.execute("""
    UPDATE fact_sales SET
        line_total = (SELECT ol.line_total FROM order_line ol WHERE ol.line_id = fact_sales.line_id),
        quantity   = (SELECT ol.quantity   FROM order_line ol WHERE ol.line_id = fact_sales.line_id),
        unit_price = (SELECT ol.unit_price FROM order_line ol WHERE ol.line_id = fact_sales.line_id)
    WHERE fiscal_year=2026 AND fiscal_month=3
""")
conn.commit()
print(f"✅ fact_sales T03 sync : {cur.rowcount:,} rows")

# ── FINAL SUMMARY ──────────────────────────────────────
cur.execute("""
    SELECT fiscal_year, fiscal_month,
           COUNT(DISTINCT so_number) don,
           SUM(line_total) doanh_thu
    FROM fact_sales
    GROUP BY fiscal_year, fiscal_month
    ORDER BY 1,2
""")
print(f"\n{'Năm-T':>8} {'Đơn':>8} {'Doanh thu (tỷ)':>16}")
print("  "+"-"*36)
for y,m,d,r in cur.fetchall():
    flag = " ← MỚI" if y==2026 and m==3 else ""
    print(f"  {y}-T{m:02d}  {d:>6,}   {r/1e9:>14.2f}{flag}")

# Tổng 2026
cur.execute("SELECT SUM(line_total) FROM fact_sales WHERE fiscal_year=2026")
total_2026 = cur.fetchone()[0]
print(f"\n  📊 Tổng 2026: {total_2026/1e9:.2f} tỷ đồng")

conn.close()

✅ sales_order updated : 1,132
✅ order_line updated  : 1,132
✅ fact_sales T03 sync : 1,132 rows

   Năm-T      Đơn   Doanh thu (tỷ)
  ------------------------------------
  2025-T01      61             6.40
  2025-T02     185            12.67
  2025-T03     447            37.16
  2026-T01     482            42.27
  2026-T02     452            38.78
  2026-T03   1,132            40.80 ← MỚI

  📊 Tổng 2026: 121.85 tỷ đồng


In [39]:
# Cell 33 — Báo cáo tổng kết FINAL
conn = sqlite3.connect(db_path)
cur = conn.cursor()

print("=" * 62)
print("  ✅ PIPELINE HOÀN CHỈNH — TỔNG KẾT")
print("=" * 62)

print("\n📥 DỮ LIỆU ĐÃ GHI:")
for t, q in [
    ("email_log",   "SELECT COUNT(*) FROM email_log"),
    ("sales_order", "SELECT COUNT(*) FROM sales_order WHERE invoice_symbol='C26TTN' AND order_date LIKE '2026-03%'"),
    ("order_line",  "SELECT COUNT(*) FROM order_line WHERE so_number IN (SELECT so_number FROM sales_order WHERE order_date LIKE '2026-03%')"),
    ("fact_sales",  "SELECT COUNT(*) FROM fact_sales WHERE fiscal_year=2026 AND fiscal_month=3"),
]:
    cur.execute(q)
    print(f"   {t:15s}: {cur.fetchone()[0]:>6,} rows")

print("\n📊 FACT_SALES TOÀN BỘ:")
cur.execute("""
    SELECT fiscal_year, fiscal_month,
           COUNT(DISTINCT so_number) don,
           SUM(line_total) dt
    FROM fact_sales GROUP BY 1,2 ORDER BY 1,2
""")
print(f"   {'Tháng':>8} {'Đơn':>8} {'Doanh thu':>18}")
print("   " + "-"*38)
total = 0
for y,m,d,r in cur.fetchall():
    flag = " ★" if y==2026 and m==3 else ""
    print(f"   {y}-T{m:02d}  {d:>6,}   {r/1e9:>14.2f} tỷ{flag}")
    total += r

print("   " + "-"*38)
print(f"   {'TOTAL':>8}         {total/1e9:>14.2f} tỷ")

print(f"""
📝 GHI CHÚ:
   • T03/2026: 1 order_line/đơn (email chỉ có tổng hợp)
   • 4 đại lý không map được customer_code → 'KH-UNKNOWN'
   • Cần PDF để bổ sung chi tiết SKU cho T03/2026
   • DB: {db_path}
""")
print("=" * 62)
conn.close()

  ✅ PIPELINE HOÀN CHỈNH — TỔNG KẾT

📥 DỮ LIỆU ĐÃ GHI:
   email_log      :  1,132 rows
   sales_order    :  1,132 rows
   order_line     :  1,132 rows
   fact_sales     :  1,132 rows

📊 FACT_SALES TOÀN BỘ:
      Tháng      Đơn          Doanh thu
   --------------------------------------
   2025-T01      61             6.40 tỷ
   2025-T02     185            12.67 tỷ
   2025-T03     447            37.16 tỷ
   2026-T01     482            42.27 tỷ
   2026-T02     452            38.78 tỷ
   2026-T03   1,132            40.80 tỷ ★
   --------------------------------------
      TOTAL                 178.09 tỷ

📝 GHI CHÚ:
   • T03/2026: 1 order_line/đơn (email chỉ có tổng hợp)
   • 4 đại lý không map được customer_code → 'KH-UNKNOWN'
   • Cần PDF để bổ sung chi tiết SKU cho T03/2026
   • DB: /tmp/tnbike_db/tnbike.db



In [40]:
# Cell 34 — Export tất cả ra CSV
import pandas as pd, sqlite3

conn = sqlite3.connect("/tmp/tnbike_db/tnbike.db")

tables = ["email_log", "sales_order", "order_line", "fact_sales"]
for t in tables:
    df_exp = pd.read_sql(f"SELECT * FROM {t}", conn)
    df_exp.to_csv(f"/tmp/tnbike_db/{t}.csv", index=False)
    print(f"✅ {t}.csv — {len(df_exp):,} rows")

conn.close()

✅ email_log.csv — 1,132 rows
✅ sales_order.csv — 2,759 rows
✅ order_line.csv — 35,194 rows
✅ fact_sales.csv — 35,194 rows


In [41]:
# Cell 35 — Download từ Colab về máy local
from google.colab import files

for t in tables:
    files.download(f"/tmp/tnbike_db/{t}.csv")

# Hoặc download cả notebook
files.download("pipeline_tnbike.ipynb")  # File > Save trước

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: pipeline_tnbike.ipynb

In [42]:
# Cell 35 — VALIDATE DATA T03/2026
conn = sqlite3.connect("/tmp/tnbike_db/tnbike.db")
cur  = conn.cursor()

print("=" * 60)
print("  🔍 VALIDATION REPORT — T03/2026")
print("=" * 60)

# ── 1. ROW COUNT ──────────────────────────────────────────
print("\n[1] ROW COUNT")
for tbl, q in [
    ("sales_order", "SELECT COUNT(*) FROM sales_order WHERE order_date LIKE '2026-03%'"),
    ("order_line",  "SELECT COUNT(*) FROM order_line  WHERE so_number IN (SELECT so_number FROM sales_order WHERE order_date LIKE '2026-03%')"),
    ("fact_sales",  "SELECT COUNT(*) FROM fact_sales  WHERE fiscal_year=2026 AND fiscal_month=3"),
]:
    cur.execute(q); n = cur.fetchone()[0]
    status = "✅" if n == 1132 else "❌"
    print(f"   {status} {tbl:15s}: {n:,}  (expected 1,132)")

# ── 2. NULL CHECK ─────────────────────────────────────────
print("\n[2] NULL / ZERO CHECK")
checks = [
    ("total_amount = 0",  "SELECT COUNT(*) FROM sales_order WHERE order_date LIKE '2026-03%' AND (total_amount IS NULL OR total_amount = 0)"),
    ("line_total = 0",    "SELECT COUNT(*) FROM order_line  WHERE so_number IN (SELECT so_number FROM sales_order WHERE order_date LIKE '2026-03%') AND (line_total IS NULL OR line_total = 0)"),
    ("customer NULL",     "SELECT COUNT(*) FROM sales_order WHERE order_date LIKE '2026-03%' AND customer_code IS NULL"),
    ("fact line_total=0", "SELECT COUNT(*) FROM fact_sales  WHERE fiscal_year=2026 AND fiscal_month=3 AND (line_total IS NULL OR line_total=0)"),
]
for label, q in checks:
    cur.execute(q); n = cur.fetchone()[0]
    status = "✅" if n == 0 else f"⚠️ ({n} rows)"
    print(f"   {status}  {label}")

# ── 3. AMOUNT SANITY ──────────────────────────────────────
print("\n[3] AMOUNT SANITY")
cur.execute("""
    SELECT MIN(total_amount), MAX(total_amount), AVG(total_amount), SUM(total_amount)
    FROM sales_order WHERE order_date LIKE '2026-03%'
""")
mn, mx, avg, total = cur.fetchone()
print(f"   Min  : {mn:>15,.0f} đ")
print(f"   Max  : {mx:>15,.0f} đ")
print(f"   Avg  : {avg:>15,.0f} đ")
print(f"   Total: {total/1e9:>14.2f} tỷ  {'✅' if 35e9 < total < 50e9 else '⚠️'}")

# ── 4. DUPLICATE CHECK ────────────────────────────────────
print("\n[4] DUPLICATE so_number")
cur.execute("""
    SELECT so_number, COUNT(*) c FROM sales_order
    WHERE order_date LIKE '2026-03%'
    GROUP BY so_number HAVING c > 1
""")
dups = cur.fetchall()
print(f"   {'✅ Không có duplicate' if not dups else f'❌ {len(dups)} duplicates: {dups[:5]}'}")

# ── 5. UNKNOWN CUSTOMER ───────────────────────────────────
print("\n[5] KH-UNKNOWN")
cur.execute("""
    SELECT customer_code, COUNT(*) FROM sales_order
    WHERE order_date LIKE '2026-03%'
    GROUP BY customer_code
    ORDER BY 2 DESC LIMIT 5
""")
print(f"   {'Code':20s} {'Đơn':>6}")
for code, cnt in cur.fetchall():
    flag = " ⚠️" if code == 'KH-UNKNOWN' else ""
    print(f"   {str(code):20s} {cnt:>6}{flag}")

# ── 6. DATE RANGE ─────────────────────────────────────────
print("\n[6] DATE RANGE T03")
cur.execute("""
    SELECT MIN(order_date), MAX(order_date)
    FROM sales_order WHERE order_date LIKE '2026-03%'
""")
mn_d, mx_d = cur.fetchone()
print(f"   Từ : {mn_d}")
print(f"   Đến: {mx_d}")

print("\n" + "=" * 60)
conn.close()

  🔍 VALIDATION REPORT — T03/2026

[1] ROW COUNT
   ✅ sales_order    : 1,132  (expected 1,132)
   ✅ order_line     : 1,132  (expected 1,132)
   ✅ fact_sales     : 1,132  (expected 1,132)

[2] NULL / ZERO CHECK
   ✅  total_amount = 0
   ✅  line_total = 0
   ✅  customer NULL
   ✅  fact line_total=0

[3] AMOUNT SANITY
   Min  :         882,777 đ
   Max  :     638,888,889 đ
   Avg  :      36,045,978 đ
   Total:          40.80 tỷ  ✅

[4] DUPLICATE so_number
   ✅ Không có duplicate

[5] KH-UNKNOWN
   Code                    Đơn
   KH-00429                 92
   KH-00669                 88
   KH-00511                 29
   KH-00566                 22
   KH-00605                 19

[6] DATE RANGE T03
   Từ : 2026-03-01
   Đến: 2026-03-31



In [44]:
import os, shutil, subprocess
from google.colab import userdata

# ── TOKEN ──────────────────────────────────────────
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
    print("✅ Token từ Colab Secrets")
except:
    TOKEN = "REMOVED"   # ← paste token vào đây nếu chưa dùng Secrets
    print("⚠️  Token hardcode")

REPO    = "trungcandygit/10tr_devided_by_-"
TMP     = "/tmp/tnbike_push"
CSV_DIR = "/tmp/tnbike_db"

# ── CẤU TRÚC THƯ MỤC CHUẨN ────────────────────────
print(f"""
📁 Cấu trúc sẽ tạo trên GitHub:
10tr_devided_by_-/
├── README.md
├── notebooks/
│   └── pipeline_clean.ipynb
└── data/
    ├── fact_sales.csv
    ├── sales_order.csv
    ├── order_line.csv
    └── email_log.csv
""")

⚠️  Token hardcode

📁 Cấu trúc sẽ tạo trên GitHub:
10tr_devided_by_-/
├── README.md
├── notebooks/
│   └── pipeline_clean.ipynb
└── data/
    ├── fact_sales.csv
    ├── sales_order.csv
    ├── order_line.csv
    └── email_log.csv



In [46]:
# Cell 37 — FIX: paste token thẳng vào (sau khi tạo token mới)
import os, shutil, subprocess, glob
from google.colab import _message

TOKEN   = "GITHUB_TOKEN_REMOVED"   # ← token mới sau khi xóa cái cũ
REPO    = "trungcandygit/10tr_devided_by_-"
TMP     = "/tmp/tnbike_push"
CSV_DIR = "/tmp/tnbike_db"

# Lưu notebook
_message.blocking_request('save_notebook', request='', timeout_sec=30)
print("✅ Notebook saved")

# Clone
shutil.rmtree(TMP, ignore_errors=True)
r = subprocess.run(f"git clone https://{TOKEN}@github.com/{REPO}.git {TMP}",
               shell=True, capture_output=True, text=True)
print("✅ Cloned" if r.returncode==0 else f"❌ {r.stderr}")

# Tạo thư mục + copy file
os.makedirs(f"{TMP}/data",      exist_ok=True)
os.makedirs(f"{TMP}/notebooks", exist_ok=True)

for f in ["fact_sales.csv","sales_order.csv","order_line.csv","email_log.csv"]:
    shutil.copy(f"{CSV_DIR}/{f}", f"{TMP}/data/{f}")
    print(f"✅ data/{f}")

nbs = glob.glob("/content/*.ipynb")
if nbs:
    shutil.copy(nbs[0], f"{TMP}/notebooks/pipeline_clean.ipynb")
    print(f"✅ notebooks/pipeline_clean.ipynb")

# Push
for cmd in [
    f"git -C {TMP} config user.name 'trungcandygit'",
    f"git -C {TMP} config user.email 'trungcandygit@users.noreply.github.com'",
    f"git -C {TMP} add .",
    f"git -C {TMP} commit -m 'init: pipeline_clean + data T03/2026'",
    f"git -C {TMP} push origin main",
]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {cmd.split(TMP+' ')[-1][:40]}")
    if r.returncode != 0: print(f"   {r.stderr.strip()}")

print(f"\n🎉 https://github.com/{REPO}")

✅ Notebook saved
✅ Cloned
✅ data/fact_sales.csv
✅ data/sales_order.csv
✅ data/order_line.csv
✅ data/email_log.csv
✅ config user.name 'trungcandygit'
✅ config user.email 'trungcandygit@users.n
✅ add .
✅ commit -m 'init: pipeline_clean + data T
✅ push origin main

🎉 https://github.com/trungcandygit/10tr_devided_by_-


In [47]:
# Kiểm tra có file .ipynb nào trong /content
import glob, os

nbs = glob.glob("/content/*.ipynb")
print("📓 Notebooks tìm thấy:")
for nb in nbs:
    size = os.path.getsize(nb)/1024
    print(f"   {nb}  ({size:.1f} KB)")

📓 Notebooks tìm thấy:


In [48]:
# Bước 1: Lưu notebook vào Drive trước
from google.colab import drive, _message
drive.mount('/content/drive')

# Lưu notebook
_message.blocking_request('save_notebook', request='', timeout_sec=30)

# Bước 2: Tìm file trong Drive
import glob
nbs = glob.glob("/content/drive/MyDrive/**/*.ipynb", recursive=True)
print("📓 Notebooks trên Drive:")
for nb in nbs:
    print(f"   {nb}")

Mounted at /content/drive
📓 Notebooks trên Drive:
   /content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled6.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled7.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled8.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled9.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled10.ipynb
   /content/drive/MyDrive/Colab Notebooks/Phần dự án.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled11.ipynb
   /content/drive/MyDrive/Colab Notebooks/Untitled12.ipynb


In [50]:
# Bước 3: Copy từ Drive vào repo rồi push
NB_SOURCE = "/content/drive/MyDrive/..."  # ← path từ kết quả trên

shutil.copy(NB_SOURCE, f"{TMP}/notebooks/pipeline_clean.ipynb")
print("✅ Copied notebook")

for cmd in [
    f"git -C {TMP} add notebooks/",
    f"git -C {TMP} commit -m 'add: pipeline_clean.ipynb'",
    f"git -C {TMP} push origin main",
]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {cmd.split(TMP+' ')[-1]}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/...'

In [51]:
# Bước 1: Mount Drive trước
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
# Bước 2: Tìm tất cả notebook trong Drive
import glob
nbs = glob.glob("/content/drive/MyDrive/**/*.ipynb", recursive=True)
for nb in nbs:
    print(nb)

/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled6.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled7.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled8.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled9.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled10.ipynb
/content/drive/MyDrive/Colab Notebooks/Phần dự án.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled11.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled12.ipynb


In [53]:
# Lưu notebook hiện tại lên Drive trước
from google.colab import _message
_message.blocking_request('save_notebook', request='', timeout_sec=30)
print("✅ Saved")

# Copy + push
NB_SOURCE = "/content/drive/MyDrive/Colab Notebooks/Untitled12.ipynb"

shutil.copy(NB_SOURCE, f"{TMP}/notebooks/pipeline_clean.ipynb")
print("✅ Copied notebook")

for cmd in [
    f"git -C {TMP} add notebooks/",
    f"git -C {TMP} commit -m 'add: pipeline_clean.ipynb'",
    f"git -C {TMP} push origin main",
]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {cmd.split(TMP+' ')[-1][:40]}")
    if r.returncode != 0: print(f"   {r.stderr.strip()}")

print(f"\n🎉 https://github.com/trungcandygit/10tr_devided_by_-")

✅ Saved
✅ Copied notebook
✅ add notebooks/
✅ commit -m 'add: pipeline_clean.ipynb'
❌ push origin main
   remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: 878d7c02c0606f67284e47491fdea1e409475c4b        
remote:            path: notebooks/pipeline_clea

In [54]:
# Lưu notebook hiện tại lên Drive trước
from google.colab import _message
_message.blocking_request('save_notebook', request='', timeout_sec=30)
print("✅ Saved")

# Copy + push
NB_SOURCE = "/content/drive/MyDrive/Colab Notebooks/Untitled12.ipynb"

shutil.copy(NB_SOURCE, f"{TMP}/notebooks/pipeline_clean.ipynb")
print("✅ Copied notebook")

for cmd in [
    f"git -C {TMP} add notebooks/",
    f"git -C {TMP} commit -m 'add: pipeline_clean.ipynb'",
    f"git -C {TMP} push origin main",
]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {cmd.split(TMP+' ')[-1][:40]}")
    if r.returncode != 0: print(f"   {r.stderr.strip()}")

print(f"\n🎉 https://github.com/trungcandygit/10tr_devided_by_-")

✅ Saved
✅ Copied notebook
✅ add notebooks/
❌ commit -m 'add: pipeline_clean.ipynb'
   
❌ push origin main
   remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: 878d7c02c0606f67284e47491fdea1e409475c4b        
remote:            path: notebooks/pipeline_

In [55]:
# Xóa notebook cũ có token, copy lại bản vừa sửa
from google.colab import _message
_message.blocking_request('save_notebook', request='', timeout_sec=30)

import shutil, subprocess

NB_SOURCE = "/content/drive/MyDrive/Colab Notebooks/Untitled12.ipynb"
TMP = "/tmp/tnbike_push"

# Overwrite bằng bản đã xóa token
shutil.copy(NB_SOURCE, f"{TMP}/notebooks/pipeline_clean.ipynb")
print("✅ Copied notebook sạch")

# Reset commit cũ (bỏ commit có token)
for cmd in [
    f"git -C {TMP} reset HEAD~1",           # undo commit có token
    f"git -C {TMP} add notebooks/",
    f"git -C {TMP} commit -m 'add: pipeline_clean.ipynb (no secrets)'",
    f"git -C {TMP} push origin main",
]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {cmd.split(TMP+' ')[-1][:45]}")
    if r.returncode != 0: print(f"   {r.stderr.strip()}")

print("\n🎉 https://github.com/trungcandygit/10tr_devided_by_-")

✅ Copied notebook sạch
✅ reset HEAD~1
✅ add notebooks/
✅ commit -m 'add: pipeline_clean.ipynb (no secr
❌ push origin main
   remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: 5b8cd1dd7895ecf63fe71475fdf98b9304acda71        
remote:            path: not